# 06 — Open-Meteo API und Kafka-Producer

## Zweck
Aktuelle Luftqualitätswerte von der **Open-Meteo Air Quality REST-API** abrufen, als Bronze-JSON
speichern, in versionierte **Events** verpacken und mit einem **Kafka-Producer** an das Topic senden.
Damit erfüllen wir zwei Anforderungen: REST-API-Quelle und Kafka als Daten-Broker.

## Event-Vertrag (schema_version 1.0)
Jedes Event hat eine deterministische `event_id` (SHA256), damit Spark später Duplikate erkennen kann:
```json
{"event_id": "...", "schema_version": "1.0", "source": "open_meteo", "city_id": "berlin_de",
 "event_time_utc": "...", "ingestion_time_utc": "...", "data_status": "live",
 "pm2_5": 12.3, "pm10": 18.7, "no2": 24.1}
```

## Ausgabe
- Bronze: `data/bronze/open_meteo_raw/<city_id>.json` (rohe API-Antwort) und `..._events.jsonl`
- Kafka-Topic: `KAFKA_TOPIC_AIR_QUALITY_LIVE`

## Konfiguration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from hashlib import sha256
import json
import os

from dotenv import load_dotenv
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=False)
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "bronze" / "open_meteo_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OPEN_METEO_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "air_quality_live")
MAX_HOURS = 24   # ein Tag stuendlicher Werte je Stadt
EVENTS_PATH = RAW_DIR / "open_meteo_air_quality_events.jsonl"

city_reference_df = pd.read_parquet(DATA_DIR / "silver" / "city_reference.parquet")
print({"kafka": KAFKA_BOOTSTRAP_SERVERS, "topic": KAFKA_TOPIC})

{'kafka': 'kafka:29092', 'topic': 'air_quality_live'}


## Open-Meteo abrufen und Events bauen
Pro Stadt holen wir die stündlichen Werte, speichern die Rohantwort und bilden bis zu `MAX_HOURS` Events.

In [2]:
import time


def fetch_city(row):
    params = {"latitude": row["latitude"], "longitude": row["longitude"],
              "hourly": "pm2_5,pm10,nitrogen_dioxide", "timezone": "UTC", "forecast_days": 1}
    # Open-Meteo schliesst bei schnellen Folge-Requests gelegentlich die Verbindung:
    # kleiner Retry, neue Verbindung pro Versuch.
    for attempt in range(3):
        try:
            resp = requests.get(OPEN_METEO_URL, params=params, timeout=20,
                                headers={"Connection": "close"})
            resp.raise_for_status()
            payload = resp.json()
            (RAW_DIR / f"{row['city_id']}.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
            return payload
        except requests.RequestException:
            if attempt == 2:
                raise
            time.sleep(2)


def build_events(row, payload):
    hourly = payload["hourly"]
    ingestion_time = datetime.now(timezone.utc).isoformat()
    events = []
    for i, event_time in enumerate(hourly["time"][:MAX_HOURS]):
        raw = f"{row['city_id']}|{event_time}|open_meteo|1.0"
        events.append({
            "event_id": sha256(raw.encode()).hexdigest(),
            "schema_version": "1.0",
            "source": "open_meteo",
            "city_id": row["city_id"],
            "event_time_utc": event_time,
            "ingestion_time_utc": ingestion_time,
            "data_status": "live",
            "pm2_5": hourly["pm2_5"][i],
            "pm10": hourly["pm10"][i],
            "no2": hourly["nitrogen_dioxide"][i],
        })
    return events


all_events = []
for _, row in city_reference_df.iterrows():
    all_events.extend(build_events(row, fetch_city(row)))
    time.sleep(1)   # hoeflich gegenueber der API

EVENTS_PATH.write_text("\n".join(json.dumps(e) for e in all_events), encoding="utf-8")
print(f"{len(all_events)} Events gebaut und nach {EVENTS_PATH.name} geschrieben.")
all_events[0]

192 Events gebaut und nach open_meteo_air_quality_events.jsonl geschrieben.


{'event_id': 'a2713ab55595a7fb4baed7a6cc66155252e2111439caf0577182f633dd4af04b',
 'schema_version': '1.0',
 'source': 'open_meteo',
 'city_id': 'vienna_at',
 'event_time_utc': '2026-06-05T00:00',
 'ingestion_time_utc': '2026-06-05T15:19:54.513237+00:00',
 'data_status': 'live',
 'pm2_5': 4.5,
 'pm10': 6.9,
 'no2': 8.7}

## Kafka-Producer
Genau wie in der Vorlesung (class5): `Producer({'bootstrap.servers': ...})`, dann jedes Event
mit `produce()` senden und mit `flush()` zustellen. Schlüssel ist die `city_id`.

In [3]:
from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS})

for event in all_events:
    producer.produce(KAFKA_TOPIC, key=event["city_id"], value=json.dumps(event))
producer.flush()

print(f"{len(all_events)} Events an Kafka-Topic '{KAFKA_TOPIC}' gesendet.")

192 Events an Kafka-Topic 'air_quality_live' gesendet.


## Nächster Schritt
Notebook `07` ausführen — Spark liest diese Events aus Kafka und schreibt sie als Parquet.